# 🛡️ Pipeline CI/CD: Gatekeeper e Retreino Automático

## 🎯 Objetivo
Atua como **Gatekeeper** (Guardião). Antes de atualizar o modelo em produção, realiza um teste rápido nos últimos meses.

## 🚨 Importante
Se o RMSE for alto, o pipeline falha propositalmente (`raise Exception`).


In [0]:
# Define a assinatura de entrada/saída
# --- 0. IMPORTS E SETUP ---
%load_ext autoreload
%autoreload 2

import sys
import os
import pickle
import pandas as pd
import numpy as np
import mlflow
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pyspark.sql.functions as F
import gzip

# Adiciona o diretório raiz ao path para importar os módulos 'src'
sys.path.append(os.getcwd())

# Imports dos Módulos do Projeto
from src.validation.config import Config
from src.validation.data import DataIngestion
from src.validation.pipeline import ProjectPipeline
from src.validation.trainer import ModelTrainer 
from src.deploy.wrapper import UnifiedForecaster
from pytorch_lightning.callbacks import EarlyStopping

# Bibliotecas de Modelagem e MLflow
from darts.models import LightGBMModel, BlockRNNModel
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

# Configurações de Otimização do Spark (Delta Lake)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

In [0]:
# ==============================================================================
# 1. CONFIGURAÇÃO DO AMBIENTE E DATAS
# ==============================================================================
config = Config(spark)

# [IMPORTANTE] Força o nome do experimento para ser o de DEPLOY
config.EXPERIMENT_NAME = "/Workspace/Shared/data_science/projetos/cvc_curva_de_vendas_por_canal/experiments/Model_Deploy_CVC_Ecommerce"

# --- CORREÇÃO AQUI ---
# Recuamos 2 meses para garantir que o horizonte de 35 dias esteja coberto pelos dados atuais (que param em 20/01)
data_final_deploy = pd.to_datetime(datetime.today().strftime("%Y-%m-01")) - relativedelta(months=2)

data_inicio_validacao = data_final_deploy - relativedelta(months=2)

config.TRAIN_END_DATE = data_final_deploy.strftime("%Y-%m-%d")
config.VAL_START_DATE = data_inicio_validacao.strftime("%Y-%m-%d")

# A ingestão continua pegando tudo o que tem disponível
data_limite_ingestao = pd.to_datetime(datetime.today()) + relativedelta(days=90)
config.INGESTION_END = data_limite_ingestao.strftime("%Y-%m-%d")

print(f"📂 EXPERIMENTO ALVO: {config.EXPERIMENT_NAME}")
print(f"⏱️ PERÍODO DE TREINO: {config.DATA_START} até {config.TRAIN_END_DATE}")
print(f"📥 PERÍODO DE INGESTÃO (Covariáveis): Até {config.INGESTION_END}")
print(f"🔍 JANELA DE TESTE (BACKTEST): {config.VAL_START_DATE} até {config.TRAIN_END_DATE}")

In [0]:
print(config.DATA_START)
print(config.VAL_START_DATE)
print(config.TRAIN_END_DATE)
print(config.INGESTION_END)

In [0]:
# ==========================================================================
# PREPARAÇÃO PARA (COVARIÁVEIS ESTENDIDAS)
# ==========================================================================

import pandas as pd
from datetime import datetime

# 1. Definição de Limites
forecast_n = 35
# Margem de segurança: input_chunk + output_chunk + folga
buffer_days = 120 
today_dt = pd.to_datetime(datetime.today().strftime("%Y-%m-%d"))
limit_covariates = today_dt + pd.Timedelta(days=buffer_days)

# 2. Ingestão com Janela Futura
ingestion = DataIngestion(spark, config)
df_spark_raw = ingestion.create_training_set()

# Filtramos o Spark permitindo datas futuras para as covariáveis
df_spark_filtered = df_spark_raw.filter(
    (F.col("data") >= config.DATA_START) & 
    (F.col("data") <= limit_covariates.strftime("%Y-%m-%d"))
)

# Suporte global (Dólar, IPCA) também estendido
df_global_support = ingestion.get_global_support()[config.DATA_START : limit_covariates.strftime("%Y-%m-%d")]

# 3. Construção dos Objetos Darts
# O DataIngestion.build_darts_objects usará o range expandido para as covariáveis
full_target_list_long, full_covariates_list = ingestion.build_darts_objects(df_spark_filtered, df_global_support)

# 4. Corte do Target (Vendas)
full_target_list = [ts.drop_after(today_dt) if ts.end_time() > today_dt else ts for ts in full_target_list_long]

print(f"✅ Target termina em: {full_target_list[0].end_time()}")
print(f"✅ Covariates terminam em: {full_covariates_list[0].end_time()} (Suficiente para BlockRNN)")

In [0]:
# Validação de Qualidade (Gatekeeper): Se o erro for alto, aborta o deploy
# ==============================================================================
# 2. INÍCIO DA EXECUÇÃO MESTRA
# ==============================================================================

EARLY_STOPPER = EarlyStopping(monitor="train_loss", patience=5, min_delta=0.001, mode='min')
mlflow.set_experiment(config.EXPERIMENT_NAME)
with mlflow.start_run(run_name=f"Pipeline_Completo_{config.VERSION}") as parent_run:
    
    print(f"🔗 Parent Run ID: {parent_run.info.run_id}")
    mlflow.log_param("pipeline_type", "auto_retrain_w_quality_gate")

    # ==========================================================================
    # 4. PREPARAÇÃO DO PIPELINE (SCALERS)
    # ==========================================================================
    pipeline = ProjectPipeline()
    val_cutoff_dt = pd.Timestamp(config.VAL_START_DATE) - pd.Timedelta(days=30)

    # Fit nos dados históricos (evita Data Leakage)
    train_subset_for_fit = [s.drop_after(val_cutoff_dt) for s in full_target_list]
    cov_subset_for_fit = [c.drop_after(val_cutoff_dt) for c in full_covariates_list]

    print("⚙️ Ajustando Scalers...")
    pipeline.fit(train_subset_for_fit, cov_subset_for_fit)

    # Transforma base completa
    scaled_series, scaled_covariates = pipeline.transform(full_target_list, full_covariates_list)
    train_series_static = [s.drop_after(val_cutoff_dt) for s in scaled_series]
    train_cov_static = [c.drop_after(val_cutoff_dt) for c in scaled_covariates]
    val_series_original = pipeline.inverse_transform(scaled_series, partial=True)

    # ==========================================================================
    # 5. VALIDAÇÃO AUTOMÁTICA (BACKTEST)
    # ==========================================================================
    model_params = {
    "lags": 12,
    "lags_future_covariates": [0,1,2,3],
    "output_chunk_length": 1,
    "multi_models":True,
    "device":"gpu",
    "random_state": 42
}
    models_dict = {"model_deploy": LightGBMModel(**model_params)}

    print(f"\n🚀 Executando Backtest (3 Meses)...")
    trainer = ModelTrainer(config, models_dict)
    trainer.train_evaluate_walkforward(
        train_series_static = train_series_static,
        train_covs_static = train_cov_static,
        full_series_scaled = scaled_series,
        full_covariates_scaled = scaled_covariates,
        val_series_original = val_series_original,
        target_pipeline = pipeline
    )
    
    # ==========================================================================
    # 6. GATEKEEPER (TRAVA DE SEGURANÇA)
    # ==========================================================================
    print("\n 👮 Verificando Qualidade do Modelo (Gatekeeper)...")
    
    # A. Lê os resultados salvos na tabela Delta pela validação acima
    # Filtra apenas pela versão atual do pipeline para não pegar lixo antigo
    df_results = spark.table("ds_dev.cvc_val.resultado_metricas_treinamento_ecommerce") \
                      .filter(F.col("versao") == config.VERSION)
    
    # B. Descobre qual foi o último mês validado (o mês mais recente)
    # try:
    #     last_month_str = df_results.select(F.max("metrica_mes")).collect()[0][0]
    # except:
    #     raise Exception("❌ ERRO CRÍTICO: Tabela de resultados vazia. Validação falhou silenciosamente?")

    #if not last_month_str:
    #    # Fallback se não encontrar mês (pode acontecer se o backtest pular todos os meses)
    #    print("⚠️ Aviso: Não foi possível identificar mês de validação. Tentando pegar o último disponível.")
    last_month_str = config.TRAIN_END_DATE[:7] # YYYY-MM

    print(f"   📅 Analisando mês de corte:")

    # C. Calcula o RMSE especificamente para este mês
    # 1) pega o valor mais recente de metrica_mes
    last_mes = df_results.agg(F.max("metrica_mes").alias("last_mes")).collect()[0]["last_mes"]

    # 2) filtra pelo mais recente
    df_last = (
        df_results
        .filter(F.col("metrica_mes") == F.lit(last_mes))
        .withColumn("real_d", F.col("real").cast("double"))
        .withColumn("prev_d", F.col("previsao").cast("double"))
        .filter(
            F.col("real_d").isNotNull() & F.col("prev_d").isNotNull() &
            ~F.isnan("real_d") & ~F.isnan("prev_d")
        )
        .withColumn("err", F.col("real_d") - F.col("prev_d"))
        .withColumn("abs_err", F.abs("err"))
    )

    # 3) métricas
    row = df_last.agg(
        F.count("*").alias("n_valid"),
        F.sqrt(F.mean(F.pow(F.col("err"), 2))).alias("rmse"),
        F.mean(F.col("abs_err")).alias("mae"),
        F.expr("percentile_approx(abs_err, 0.5)").alias("median_ae"),
        F.expr("percentile_approx(abs_err, 0.95)").alias("p95_abs_err")
    ).collect()[0]

    metrics = row.asDict()
    metrics["metrica_mes_usada"] = last_mes
    rmse_check = metrics['rmse']
    # Se der Nulo (sem dados), assume erro
    if rmse_check is None: rmse_check = 10000.0
    
    print(f"   📉 RMSE Calculado: {rmse_check:.4f}")
    
    # Loga na Run Pai para auditoria
    mlflow.log_metric("gatekeeper_last_month_rmse", rmse_check)

    # D. APLICA A TRAVA
    LIMIT_RMSE = 10000
    
    if rmse_check >= LIMIT_RMSE:
        error_msg = (f"⛔ BLOQUEIO DE DEPLOY: RMSE do último mês ({rmse_check:.2f}) "
                     f"excedeu o limite aceitável ({LIMIT_RMSE}). Pipeline Abortado.")
        # Marca a run pai como FALHA explicitamente
        mlflow.set_tag("pipeline_status", "BLOCKED_BY_QUALITY")
        raise Exception(error_msg)
    
    print("✅ Critério de Qualidade Aprovado! Prosseguindo para Deploy.")

    # ==========================================================================
    # 7. TREINAMENTO FINAL (FULL DATASET)
    # ==========================================================================

    print("\n 🏋️ Iniciando Treinamento Final (Full Dataset)...")
    final_model = LightGBMModel(**model_params)
    final_model.fit(scaled_series, future_covariates=scaled_covariates)

    # ==========================================================================
    # 8. REGISTRO E DEPLOY
    # ==========================================================================

    catalog_model_name = f"{config.CATALOG}.cvc_pred.cvc_ecommerce_forecast_production"
    
    print(f"💾 Registrando modelo: {catalog_model_name}")
    
    # Tags de rastreabilidade
    mlflow.set_tag("parent_run_id", parent_run.info.run_id)
    mlflow.set_tag("quality_check_rmse", f"{rmse_check:.2f}")

    # Preparação de Artefatos (Pipeline, Modelo, Metadados)
    sample_ts = full_target_list[0]
    sample_cov = full_covariates_list[0]
    
    training_metadata = {
        "static_cols_order": [c for c in sample_ts.static_covariates.columns.tolist() if c != "codigo_loja"],
        "covariate_cols_order": sample_cov.components.tolist(),
        "max_lag": 15
    }

    pipeline_path, model_path,  meta_path = "pipeline.pkl", "lgbm_model.pkl", "model_metadata.pkl"
    
    with open(pipeline_path, "wb") as f: pickle.dump(pipeline, f)
    with open(model_path, "wb") as f: pickle.dump(final_model, f)
    with open(meta_path, "wb") as f: pickle.dump(training_metadata, f)
    artifacts = {"pipeline": pipeline_path, "darts_model": model_path, "metadata": meta_path}

    # Assinatura
    market_cols = [col for col in df_global_support.columns]
    full_input_dict = {
        **{col: [0.0] for col in market_cols},
        "data": ["2025-01-01"], "codigo_loja": ["1"], "target_vendas": [1000.0],
        "n": [35], "is_feriado": [0.0], "cluster_loja": ["A"], "sigla_uf": ["SP"], "tipo_loja": ["SHOPPING"], "modelo_loja": ["PADRAO"]
    }
    input_example = pd.DataFrame(full_input_dict)
    output_example = pd.DataFrame({"data_previsao": ["2025-01-02"], "previsao_venda": [1050.0], "codigo_loja": ["1"]})
    signature = infer_signature(input_example, output_example)

    # Log do Modelo
    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=UnifiedForecaster(), 
        artifacts=artifacts,
        input_example=input_example,
        signature=signature,
        metadata={"description": "lightgbm"}, 
        registered_model_name=catalog_model_name
    )
    
    # Promoção @Champion
    client = MlflowClient()
    mv = model_info.registered_model_version
    client.set_registered_model_alias(name=catalog_model_name, alias="Champion", version=mv)
    
    client.update_model_version(
        name=catalog_model_name, version=mv,
        description=f"ecommerce_lightgbm"
    )

    for p in [pipeline_path, model_path,  meta_path]:
        if os.path.exists(p): os.remove(p)

print(f"\n✨ Pipeline Finalizado com Sucesso! RMSE Validado: {rmse_check:.2f}")